# 02 — CNN Baseline (Genre Multi-label)

Train a compact CNN on log-mel for **multi-label genre**.

- Split: official **split-0** only (never a random split)
- Metrics: NaN-safe macro ROC-AUC and PR-AUC (undefined tags are **excluded**, not zeroed)
- Checkpoint: **best validation** macro PR-AUC (`best_macro_map` is updated inside the save branch)

Paper reference (full set): **0.7260 ROC-AUC / 0.1592 PR-AUC** — shard subset will differ.


## Kaggle setup (every notebook)

### A. Settings
1. Right sidebar → **Internet → On** (required for downloads).
2. **GPU**: Off for `00`/`01`/`04`–`06`. **GPU (T4)** on for `02`/`03`/`07`.

### B. How data moves (do not skip)
Kaggle **does not** keep `/kaggle/working` when you open a *new* notebook.

**After notebook 00 finishes:**
1. **Save Version** (top-right) → **Save & Run All** (or Quick Save if already finished).
2. Open **Advanced** → tick **Always save output**.
3. Wait until the version is **Success**.
4. Note the kernel slug (yours is **`thevifernando/dnn-download-data-1`**).

**In the next notebook (01, then 02, …):**
1. **Add Input** (right sidebar) → **Your notebooks** / **Notebook Output**.
2. Select **`dnn-download-data-1`** (latest successful version).
3. Files appear at `/kaggle/input/dnn-download-data-1/` (**read-only**).
4. This bootstrap **reads mels from that input** (does **not** copy 10 shards — they would overflow disk).
5. It **writes** new files (manifest, features, checkpoints) to `/kaggle/working/MTG_Instrument`.
6. **Save Version + save output** again so the *next* notebook can **Add Input** *this* notebook too (chain: 00 → 01 → 02 …).

### C. CLI (laptop only — not needed on Kaggle)
```bash
kaggle kernels output thevifernando/dnn-download-data-1 -p ./from_00
```
On Kaggle you **Add Input** instead of this command.

### D. GitHub
Commit **notebooks only** to `thevindu-branch`. Do **not** git-push the `.npy` shards (too large). Data stays on Kaggle output.


## Step 0 — Packages + GPU

Enable **GPU** (T4) in Settings for this notebook.


In [ ]:
!pip install -q scikit-learn tqdm


## Step 1 — Bootstrap paths (find manifest + annotations)


In [ ]:
from pathlib import Path
import os, json, random, re, shutil, socket, urllib.request
import numpy as np
import pandas as pd

KERNEL_SLUG = "dnn-download-data-1"  # notebook 00 Kaggle slug — change if yours differs
WORKING_ROOT = Path("/kaggle/working/MTG_Instrument")
INPUT_BASE = Path("/kaggle/input")
RAW_ANN = "https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data"
NEEDED_ANN = [
    "splits/split-0/autotagging_genre-train.tsv",
    "splits/split-0/autotagging_genre-validation.tsv",
    "splits/split-0/autotagging_genre-test.tsv",
    "splits/split-0/autotagging_instrument-train.tsv",
    "splits/split-0/autotagging_instrument-validation.tsv",
    "splits/split-0/autotagging_instrument-test.tsv",
    "autotagging_genre.tsv",
    "autotagging_instrument.tsv",
]
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


def check_internet(host: str = "github.com", port: int = 443, timeout: float = 5) -> bool:
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False


def normalize_track_id(raw) -> str | None:
    """MTG ids are 7-digit zero-padded (track_0000948 → 0000948)."""
    m = re.search(r"(\d+)", str(raw))
    if not m:
        return None
    return f"{int(m.group(1)):07d}"


def _find_file(name: str, bases: list[Path]) -> Path | None:
    for base in bases:
        if not base.exists():
            continue
        hits = list(base.rglob(name))
        if hits:
            return hits[0]
    return None


def discover_input_root() -> Path | None:
    """Find a previous notebook-00 output or MTG data folder under /kaggle/input."""
    if not INPUT_BASE.exists():
        return None
    for marker in [
        "song_manifest.csv",
        "autotagging_genre-train.tsv",
        "autotagging_genre.tsv",
        ".shard_00_done",
    ]:
        hit = _find_file(marker, [INPUT_BASE])
        if hit is None:
            continue
        if marker == "song_manifest.csv":
            return hit.parents[1]  # .../MTG_Instrument/dataset/song_manifest.csv
        if marker == "autotagging_genre-train.tsv":
            # .../annotations/splits/split-0/file  OR  .../data/splits/split-0/file
            p = hit
            for _ in range(6):
                if (p / "dataset").exists() or p.name in {"MTG_Instrument", "data"}:
                    return p if p.name != "data" else p
                p = p.parent
            return hit.parents[2]
        if marker == "autotagging_genre.tsv":
            parent = hit.parent
            if parent.name == "annotations":
                return parent.parent
            return parent  # MTG data/
        if marker == ".shard_00_done":
            return hit.parents[2]  # .../MTG_Instrument/dataset/logmel_songs/.shard
    for p in INPUT_BASE.rglob("MTG_Instrument"):
        if p.is_dir():
            return p
    return None


def find_mel_dir() -> Path:
    """Prefer attached kernel output (read-only). Never copy 10 shards into working."""
    bases = [
        Path(f"/kaggle/input/{KERNEL_SLUG}") / "MTG_Instrument" / "dataset" / "logmel_songs",
        Path(f"/kaggle/input/{KERNEL_SLUG}") / "dataset" / "logmel_songs",
        WORKING_ROOT / "dataset" / "logmel_songs",
    ]
    kernel = Path(f"/kaggle/input/{KERNEL_SLUG}")
    extra = []
    if INPUT_BASE.exists():
        extra.append(INPUT_BASE)
    if kernel.exists():
        extra.append(kernel)
    for b in bases:
        if b.exists() and next(b.rglob("*.npy"), None) is not None:
            return b
    for b in extra:
        hit = next(b.rglob("*.npy"), None) if b.exists() else None
        if hit is None:
            continue
        p = hit.parent
        for _ in range(6):
            if p.name == "logmel_songs":
                return p
            p = p.parent
        return hit.parent
    return WORKING_ROOT / "dataset" / "logmel_songs"


def ensure_annotations(ann_dir: Path) -> Path:
    """Make sure split-0 TSVs exist; wget them if this is a fresh Kaggle session."""
    train = ann_dir / "splits" / "split-0" / "autotagging_genre-train.tsv"
    if train.exists():
        return ann_dir

    # maybe files are flat, or under /kaggle/input with a different layout
    hit = _find_file("autotagging_genre-train.tsv", [ann_dir, INPUT_BASE, Path("/kaggle/working")])
    if hit is not None:
        dest = ann_dir / "splits" / "split-0" / hit.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        if hit.resolve() != dest.resolve():
            shutil.copy2(hit, dest)
        # copy sibling split files from the same folder
        for name in [
            "autotagging_genre-validation.tsv",
            "autotagging_genre-test.tsv",
            "autotagging_instrument-train.tsv",
            "autotagging_instrument-validation.tsv",
            "autotagging_instrument-test.tsv",
        ]:
            sib = hit.parent / name
            if sib.exists():
                shutil.copy2(sib, dest.parent / name)
        genre_full = _find_file("autotagging_genre.tsv", [hit.parents[2] if len(hit.parents) > 2 else hit.parent, INPUT_BASE])
        if genre_full:
            shutil.copy2(genre_full, ann_dir / "autotagging_genre.tsv")
        inst_full = _find_file("autotagging_instrument.tsv", [hit.parents[2] if len(hit.parents) > 2 else hit.parent, INPUT_BASE])
        if inst_full:
            shutil.copy2(inst_full, ann_dir / "autotagging_instrument.tsv")
        print("Recovered split files from", hit.parent)
        return ann_dir

    if not check_internet():
        raise FileNotFoundError(
            "Split TSVs not found and Internet is OFF.\n"
            "Do ONE of:\n"
            "  A) Settings → Internet → On, re-run this cell (auto-download)\n"
            "  B) Add Data → attach notebook-00 output dataset (mtg-instrument-cache)\n"
            "  C) Stay in the SAME Kaggle session after running notebook 00"
        )

    print("Split TSVs missing — downloading official MTG annotations...")
    n = 0
    for rel in NEEDED_ANN:
        dest = ann_dir / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        url = f"{RAW_ANN}/{rel}"
        print("  wget", url)
        urllib.request.urlretrieve(url, dest)
        n += 1
    print(f"Downloaded {n} annotation files into {ann_dir}")
    return ann_dir


def load_split_ids(split: str, subset: str = "genre") -> set[str]:
    candidates = [
        ANN_DIR / "splits" / "split-0" / f"autotagging_{subset}-{split}.tsv",
        ANN_DIR / f"autotagging_{subset}-{split}.tsv",
        ANN_DIR / "splits" / "split-0" / f"{split}.tsv",
        ANN_DIR / f"{split}.tsv",
    ]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        found = _find_file(f"autotagging_{subset}-{split}.tsv", [ANN_DIR, INPUT_BASE, Path("/kaggle/working")])
        path = found
    if path is None:
        raise FileNotFoundError(
            f"No split file for {subset}/{split}.\n"
            "Re-run the bootstrap cell after enabling Internet, or attach notebook-00 output."
        )
    df = pd.read_csv(path, sep="\t")
    col = "TRACK_ID" if "TRACK_ID" in df.columns else df.columns[0]
    ids = set()
    for v in df[col].astype(str):
        tid = normalize_track_id(v)
        if tid:
            ids.add(tid)
    print(f"{split:12s}  {len(ids):6d} ids   ← {path}")
    return ids


ONLINE = check_internet()
print("Internet reachable:", ONLINE)
print("KERNEL_SLUG =", KERNEL_SLUG)
print("/kaggle/input folders:", list(INPUT_BASE.iterdir()) if INPUT_BASE.exists() else "n/a")

ROOT = WORKING_ROOT
ROOT.mkdir(parents=True, exist_ok=True)
MEL_DIR = find_mel_dir()
ANN_DIR = ROOT / "annotations"
# if annotations only exist on the attached kernel, point there (read-only is OK)
for cand in [
    Path(f"/kaggle/input/{KERNEL_SLUG}") / "MTG_Instrument" / "annotations",
    Path(f"/kaggle/input/{KERNEL_SLUG}") / "annotations",
]:
    if (cand / "splits" / "split-0" / "autotagging_genre-train.tsv").exists():
        ANN_DIR = cand
        break
FEAT_DIR = ROOT / "features"
CKPT_DIR = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
MANIFEST = ROOT / "dataset" / "song_manifest.csv"
att_manifest = _find_file("song_manifest.csv", [INPUT_BASE, Path("/kaggle/working")])
if not MANIFEST.exists() and att_manifest is not None:
    MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    try:
        shutil.copy2(att_manifest, MANIFEST)
        print("Copied song_manifest.csv from", att_manifest)
    except OSError:
        MANIFEST = att_manifest

for p in [ROOT / "dataset", ROOT / "annotations", FEAT_DIR, CKPT_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Small TSVs: copy/wget into working. Large mels stay on /kaggle/input.
ANN_DIR = ensure_annotations(ROOT / "annotations")

print("ROOT     =", ROOT)
print("MEL_DIR  =", MEL_DIR, "npy=", len(list(MEL_DIR.rglob('*.npy'))))
print("ANN_DIR  =", ANN_DIR)
print("split-0 train exists:", (ANN_DIR / "splits/split-0/autotagging_genre-train.tsv").exists())
print("MANIFEST =", MANIFEST, "exists=", MANIFEST.exists())


## Step 2 — Load manifest and genre multi-hot labels

Parses `autotagging_genre.tsv` (and split files) into a binary matrix `Y`.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

if not MANIFEST.exists():
    raise FileNotFoundError("song_manifest.csv missing — run notebook 01 first (same session or attach its output).")
manifest = pd.read_csv(MANIFEST)
manifest["song_id"] = manifest["song_id"].astype(str).map(lambda s: normalize_track_id(s) or s)
assert set(manifest["split"]) <= {"train", "validation", "test"}

def load_genre_multihot(song_ids: list[str]):
    candidates = [
        ANN_DIR / "autotagging_genre.tsv",
        ANN_DIR / "splits" / "split-0" / "autotagging_genre-train.tsv",
    ]
    candidates += list(ANN_DIR.rglob("*genre*.tsv"))
    tag_to_idx, rows = {}, {sid: set() for sid in song_ids}
    for path in candidates:
        if not path.exists():
            continue
        df = pd.read_csv(path, sep="\t")
        id_col = "TRACK_ID" if "TRACK_ID" in df.columns else df.columns[0]
        tag_col = "TAGS" if "TAGS" in df.columns else df.columns[-1]
        for _, r in df.iterrows():
            sid = normalize_track_id(r[id_col])
            if sid not in rows:
                continue
            raw = r[tag_col]
            if pd.isna(raw):
                continue
            for tag in str(raw).replace("|", "\t").split("\t"):
                leaf = tag.strip().split("/")[-1].split("---")[-1]
                if not leaf or leaf.lower() in {"nan", "none", "tags"}:
                    continue
                if leaf not in tag_to_idx:
                    tag_to_idx[leaf] = len(tag_to_idx)
                rows[sid].add(leaf)
        if tag_to_idx:
            print("Parsed genre tags from", path, "n_tags=", len(tag_to_idx))
            break
    if not tag_to_idx:
        raise RuntimeError("Could not parse genre TSV — re-run bootstrap / notebook 00")
    names = [None] * len(tag_to_idx)
    for t, i in tag_to_idx.items():
        names[i] = t
    Y = np.zeros((len(song_ids), len(names)), dtype=np.float32)
    id_to_row = {s: i for i, s in enumerate(song_ids)}
    for sid, tags in rows.items():
        i = id_to_row[sid]
        for t in tags:
            Y[i, tag_to_idx[t]] = 1.0
    return Y, names

song_ids = manifest["song_id"].astype(str).tolist()
Y, TAG_NAMES = load_genre_multihot(song_ids)
print("Y shape:", Y.shape, "positive rate:", float(Y.mean()))
(RESULTS_DIR / "genre_tags.json").write_text(json.dumps(TAG_NAMES, indent=2))


## Step 3 — Dataset / loaders (split-0 only)

Validation is **never** merged with test.


In [ ]:
class MelGenreDataset(Dataset):
    def __init__(self, df: pd.DataFrame, Y: np.ndarray, id_to_idx: dict, max_windows: int = 12):
        self.df = df.reset_index(drop=True)
        self.Y = Y
        self.id_to_idx = id_to_idx
        self.max_windows = max_windows

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        x = np.load(row["mel_abs"])
        if x.ndim == 2:
            x = x[None, ...]
        W = x.shape[0]
        if W >= self.max_windows:
            x = x[: self.max_windows]
        else:
            pad = np.zeros((self.max_windows - W, *x.shape[1:]), dtype=x.dtype)
            x = np.concatenate([x, pad], axis=0)
        x_mean = x.mean(axis=0, keepdims=True)
        y = self.Y[self.id_to_idx[str(row["song_id"])]]
        return torch.tensor(x_mean, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

id_to_idx = {s: i for i, s in enumerate(song_ids)}

def make_loader(split: str, bs: int = 16, shuffle=False):
    sub = manifest[manifest["split"] == split]
    assert set(sub["split"].unique()) == {split}, "split leakage"
    ds = MelGenreDataset(sub, Y, id_to_idx)
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=2, pin_memory=torch.cuda.is_available())

train_loader = make_loader("train", shuffle=True)
val_loader = make_loader("validation")
test_loader = make_loader("test")
print({s: int((manifest.split == s).sum()) for s in ["train", "validation", "test"]})


## Step 4 — Model


In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, n_tags: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, n_tags),
        )

    def forward(self, x):
        return self.head(self.features(x))

model = BaselineCNN(n_tags=Y.shape[1]).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()
print(model)


## Step 5 — Train; save **best val** checkpoint

`best_macro_map` is assigned **inside** the `if val improves` branch (Stage 1 / baseline bug-fix).


In [ ]:
def nan_safe_macro_auc(y_true, y_prob, kind="roc"):
    scores = []
    for k in range(y_true.shape[1]):
        if y_true[:, k].sum() in (0, len(y_true)):
            continue
        try:
            if kind == "roc":
                scores.append(roc_auc_score(y_true[:, k], y_prob[:, k]))
            else:
                scores.append(average_precision_score(y_true[:, k], y_prob[:, k]))
        except ValueError:
            continue
    return float(np.mean(scores)) if scores else float("nan")


@torch.no_grad()
def evaluate(loader):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        x = x.to(DEVICE)
        prob = torch.sigmoid(model(x)).cpu().numpy()
        ys.append(y.numpy())
        ps.append(prob)
    y_true, y_prob = np.concatenate(ys), np.concatenate(ps)
    return {"macro_roc_auc": nan_safe_macro_auc(y_true, y_prob, "roc"),
            "macro_pr_auc": nan_safe_macro_auc(y_true, y_prob, "pr")}


def train_one_epoch(loader):
    model.train()
    total = 0.0
    for x, y in tqdm(loader, leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        opt.step()
        total += loss.item() * len(x)
    return total / len(loader.dataset)

EPOCHS = 10
best_macro_map = 0.0
ckpt_dir = CKPT_DIR / "baseline"
ckpt_dir.mkdir(parents=True, exist_ok=True)
history = []

for epoch in range(1, EPOCHS + 1):
    tr_loss = train_one_epoch(train_loader)
    val_m = evaluate(val_loader)
    history.append({"epoch": epoch, "train_loss": tr_loss, **val_m})
    print(f"epoch {epoch}: loss={tr_loss:.4f} val_roc={val_m['macro_roc_auc']:.4f} val_pr={val_m['macro_pr_auc']:.4f}")
    if val_m["macro_pr_auc"] > best_macro_map:
        best_macro_map = val_m["macro_pr_auc"]
        torch.save({"model": model.state_dict(), "tags": TAG_NAMES, "best_macro_map": best_macro_map, "epoch": epoch},
                   ckpt_dir / "best.pt")
        print("  ✓ saved best checkpoint @", best_macro_map)

state = torch.load(ckpt_dir / "best.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(state["model"])
test_m = evaluate(test_loader)
print("TEST (split-0 only):", test_m)
pd.DataFrame(history).to_csv(RESULTS_DIR / "02_baseline_history.csv", index=False)
(RESULTS_DIR / "02_baseline_test.json").write_text(json.dumps(test_m, indent=2))
